# Post-processed Yahoo CSVs

Loads each CSV via `yahoo_csv_postprocess` and shows shape, index, column names, and a preview.

**Kernel working directory:** project root (`finance_database/`) or this folder (`yahoo_data/`). The next cell fixes `import` paths and **`importlib.reload`s** `yahoo_csv_postprocess` so edits to that file show up without restarting the kernel.

**`yahoo_key_statistics.csv`:** output is long format (MultiIndex + `category`, `sub_category`, `value`); the preview uses `reset_index()` so columns match `symbol`, `timestamp`, `url`, `status`, `category`, `sub_category`, `value`.

In [22]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

_here = Path.cwd()
_candidates = [
    _here / "yahoo_data",
    _here,
    _here.parent / "yahoo_data",
]
for p in _candidates:
    if (p / "yahoo_csv_postprocess.py").is_file():
        sys.path.insert(0, str(p.resolve()))
        break

import importlib

import yahoo_csv_postprocess

importlib.reload(yahoo_csv_postprocess)
from yahoo_csv_postprocess import POSTPROCESS_BY_FILENAME

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

# How many rows to preview per file
HEAD_ROWS = 5

# Optional: restrict to these basenames, or None for all
ONLY_FILES: list[str] | None = None  # e.g. ["yahoo_quote_metrics.csv"]

print("POSTPROCESS_BY_FILENAME:", sorted(POSTPROCESS_BY_FILENAME))

POSTPROCESS_BY_FILENAME: ['index_levels.csv', 'index_symbol_map.csv', 'yahoo_crawl_log.csv', 'yahoo_indicators.csv', 'yahoo_key_statistics.csv', 'yahoo_quote_metrics.csv', 'yahoo_timeseries.csv']


In [33]:
def show_df(name: str, df: pd.DataFrame, head_rows: int = HEAD_ROWS) -> None:
    display(Markdown(f"### {name}"))
    print(f"shape: {df.shape}")
    print(f"index: {df.index.names} ({type(df.index).__name__})")
    cols = list(df.columns)
    if len(cols) > 20:
        print(f"columns ({len(cols)}): {cols[:20]} ... (+{len(cols) - 20} more)")
    else:
        print(f"columns ({len(cols)}): {cols}")
    display(Markdown(f"**head({head_rows})**"))
    preview = df.sample(min(head_rows, len(df)))
    if name == "yahoo_key_statistics.csv":
        display(
            Markdown(
                "_Long format: index is `(symbol, timestamp, url, status)`. "
                "Preview below uses `reset_index()` so each row shows all fields as columns._"
            )
        )
        flat = preview.reset_index()
        display(flat)
    else:
        display(preview)

In [34]:
names = sorted(POSTPROCESS_BY_FILENAME)
if ONLY_FILES is not None:
    wanted = {Path(f).name for f in ONLY_FILES}
    names = [n for n in names if n in wanted]
    missing = wanted - set(names)
    if missing:
        raise ValueError(f"Unknown or unregistered: {sorted(missing)}")

for csv_name in names:
    fn = POSTPROCESS_BY_FILENAME[csv_name]
    try:
        df = fn()
    except FileNotFoundError as e:
        display(Markdown(f"### {csv_name}"))
        print(f"skip (file not found): {e}")
        continue
    except Exception as e:
        display(Markdown(f"### {csv_name}"))
        print(f"ERROR: {e!r}")
        continue
    show_df(csv_name, df)
    print()

### index_levels.csv

shape: (25958, 7)
index: ['symbol', 'date'] (MultiIndex)
columns (7): ['level_close', 'level_open', 'level_high', 'level_low', 'total_return_level', 'source_url', 'source']


**head(5)**

,,level_close,level_open,level_high,level_low,total_return_level,source_url,source
symbol,date,,,,,,,
^HGX,2017-07-01,283.5,284,287.3,279.5,NaN,https://query1.finance.yahoo.com/v8/...,yahoo
AUDUSD=X,2013-07-31,0.8902,0.8956,0.9248,0.8851,NaN,https://query1.finance.yahoo.com/v8/...,yahoo
^BSESN,2016-02-29,2.534e+04,2.315e+04,2.548e+04,2.313e+04,NaN,https://query1.finance.yahoo.com/v8/...,yahoo
AAPL,2012-06-01,23.76,20.33,24.32,19.59,NaN,https://query1.finance.yahoo.com/v8/...,yahoo
VTV,2022-04-01,140.7,148.4,151.9,140.5,NaN,https://query1.finance.yahoo.com/v8/...,yahoo


### index_symbol_map.csv

...

,,price,change,change_percent,prev_close,open,volume,...,bid_size,ask_price,ask_size,forward_dividend,forward_yield_percent,market_cap_intraday_parsed
symbol,timestamp,,,,,,,,,,,,,
AAPL,2026-03-18 08:19:32.886573,254.2,1.41,0.5577,252.8,253.1,27556024,...,200,265.6,100,1.04,0.41,3.737e+12
NVDA,2026-03-18 08:51:01.240723,183.2,2.97,1.648,180.2,183,215647512,...,100,183.4,500,0.04,0.02,4.453e+12
AAPL,2026-03-19 07:11:47.928483,249.9,-4.29,-1.687,254.2,252.6,34652568,...,100,250.1,200,1.04,0.42,3.674e+12


### yahoo_timeseries.csv

shape: (25960, 19)
index: ['symbol', 'date'] (MultiIndex)
columns (19): ['level_open', 'level_high', 'level_low', 'level_close', 'total_return_level', 'ma_50', 'ma_200', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_mid', 'bb_lower', 'stoch_k', 'stoch_d', 'source_url', 'source', 'technical_source_url']


**head(5)**

,,level_open,level_high,level_low,level_close,total_return_level,ma_50,...,bb_lower,stoch_k,stoch_d,source_url,source,technical_source_url
symbol,date,,,,,,,,,,,,,
AED=X,2010-03-31,3.672,3.673,3.672,3.673,NaN,3.671,...,3.67,92.75,90.15,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
IVV,2026-02-01,692.7,700.3,678.8,689.4,NaN,511.9,...,521.2,94.66,95.76,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
TSM,2021-01-01,111.5,136.1,110.4,121.5,NaN,48.94,...,19.15,84.36,91.43,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
CL=F,2025-11-01,61.4,61.5,57.1,58.55,NaN,76.65,...,55.17,11.67,18.75,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
MSFT,1999-03-01,37.39,47.81,36.77,40.34,NaN,6.748,...,-7.239,82.56,86.17,https://query1.finance.yahoo.com/v8/...,yahoo,https://query1.finance.yahoo.com/v8/...
